<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/03_Advanced/L33_Distributed_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L33: Distributed Training - Data and Model Parallelism

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Advanced  
**Lesson:** 33 of 45

---

## Learning Objectives

By the end of this lesson, you will:
- Use HuggingFace Trainer with multi-GPU (data parallelism)
- Understand DDP and how to launch distributed scripts
- Get an overview of model parallelism (pipeline/tensor) and DeepSpeed

---

## Concept: Distributed Training

**Data parallelism**: same model on each GPU, different batch shards; gradients all-reduced. **Model parallelism**: split model across GPUs (pipeline or tensor parallel). HuggingFace Trainer uses PyTorch DDP when multiple GPUs are visible. DeepSpeed adds ZeRO and other optimizations.

---

In [ ]:
!pip install transformers torch accelerate -q
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')
print(f"CUDA devices: {torch.cuda.device_count()}")

## Part 1: Trainer with Multi-GPU (Data Parallel)

When you run with multiple GPUs, Trainer uses PyTorch DDP automatically.

---

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

train_ds = load_dataset("imdb", split="train", trust_remote_code=True).select(range(200))
train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
train_ds.set_format("torch")

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
args = TrainingArguments(
    output_dir="./out_ddp_l33",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    dataloader_num_workers=0,
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds)
trainer.train()
print("On multi-GPU, Trainer uses DDP; each GPU gets a shard of the batch.")

## Part 2: Launching Distributed (CLI)

---

In [ ]:
# From command line, launch with:
# torchrun --nproc_per_node=2 your_script.py
# or: python -m torch.distributed.run --nproc_per_node=2 your_script.py
# Trainer automatically detects world_size and rank when run under torchrun.
print("Use: torchrun --nproc_per_node=N script.py for N GPUs.")

## Part 3: DeepSpeed (Overview)

---

In [ ]:
# DeepSpeed: pass --deepspeed ds_config.json to Trainer.
# ZeRO stage 1: optimizer states sharded
# ZeRO stage 2: + gradients sharded
# ZeRO stage 3: + parameters sharded (largest memory savings)
print("DeepSpeed ZeRO reduces memory per GPU; use TrainingArguments(deepspeed='path/to/config.json')")

## Exercises

1. Run the same training with torchrun on 2 GPUs and compare time vs 1 GPU.
2. Create a minimal DeepSpeed config (ZeRO-2) and run Trainer with it.
3. Read about pipeline parallelism (e.g. split model layers across devices).

---

## Key Takeaways

1. Data parallelism: Trainer + multiple GPUs = DDP; no code change.
2. Launch with torchrun or accelerate launch for distributed.
3. DeepSpeed/ZeRO for very large models; model parallelism for single-model-too-big.

---

## Next Lesson

**L34: Mixed Precision Training** – FP16/BF16 and AMP.

---